In [65]:
import re

def preprocess_text(text):
    text = text.lower() # lower case
    text = re.sub(r"[^a-z\s]", "", text)  # remove punctuation
    tokens = text.split() # splitting
    return tokens

In [66]:
import pandas as pd

df = pd.read_csv("actual_dataset.csv")

dataset = list(df.itertuples(index=False))

In [67]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [68]:
texts = [preprocess_text(x[0]) for x in dataset]
labels = [x[1] for x in dataset]

vectorizer = TfidfVectorizer(analyzer=lambda x: x) # forces vectorizer to read text as already tokenized
X = vectorizer.fit_transform(texts)

In [69]:
from sklearn.model_selection import train_test_split

# splitting into train, temp
X_train, X_temp, y_train, y_temp = train_test_split(X, labels, test_size=0.3, random_state=42, stratify=labels)

# splitting temp into validation, test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

print("\nDistribution of labels in each set:")
print(f"Training labels:\n{pd.Series(y_train).value_counts(normalize=True)}")
print(f"\nValidation labels:\n{pd.Series(y_val).value_counts(normalize=True)}")
print(f"\nTest labels:\n{pd.Series(y_test).value_counts(normalize=True)}")

Training set size: 127 samples
Validation set size: 27 samples
Test set size: 28 samples

Distribution of labels in each set:
Training labels:
DUCK          0.118110
RUN           0.118110
JUMP          0.118110
MOVE_LEFT     0.118110
FIRE          0.118110
PAUSE         0.118110
MOVE_RIGHT    0.110236
STOP          0.110236
ENTER_PIPE    0.070866
Name: proportion, dtype: float64

Validation labels:
MOVE_RIGHT    0.148148
JUMP          0.111111
STOP          0.111111
RUN           0.111111
FIRE          0.111111
MOVE_LEFT     0.111111
PAUSE         0.111111
DUCK          0.111111
ENTER_PIPE    0.074074
Name: proportion, dtype: float64

Test labels:
STOP          0.142857
MOVE_LEFT     0.142857
RUN           0.107143
FIRE          0.107143
DUCK          0.107143
PAUSE         0.107143
JUMP          0.107143
MOVE_RIGHT    0.107143
ENTER_PIPE    0.071429
Name: proportion, dtype: float64


## Training

In [70]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(C=1, penalty='l2', random_state=42, solver='liblinear', max_iter=1000)
model.fit(X_train, y_train)

def predict(text):
    processed_text = preprocess_text(text)
    x = vectorizer.transform([processed_text])

    probs = model.predict_proba(x)[0]
    best_idx = probs.argmax()

    label = model.classes_[best_idx]
    confidence = probs[best_idx]

    return label, confidence

print("Model trained on the training set.")

Model trained on the training set.


In [71]:
from sklearn.metrics import classification_report

train_predictions = model.predict(X_train)
report_train = classification_report(y_train, train_predictions, target_names=model.classes_)
print("Classification Report (Training Data):\n", report_train)

Classification Report (Training Data):
               precision    recall  f1-score   support

        DUCK       0.94      1.00      0.97        15
  ENTER_PIPE       1.00      0.89      0.94         9
        FIRE       1.00      1.00      1.00        15
        JUMP       1.00      1.00      1.00        15
   MOVE_LEFT       1.00      1.00      1.00        15
  MOVE_RIGHT       1.00      0.93      0.96        14
       PAUSE       1.00      1.00      1.00        15
         RUN       0.94      1.00      0.97        15
        STOP       1.00      1.00      1.00        14

    accuracy                           0.98       127
   macro avg       0.99      0.98      0.98       127
weighted avg       0.99      0.98      0.98       127



## Validation

In [72]:
import numpy as np

C_values = np.logspace(-4, 2, 7)

train_accuracies = []
val_accuracies = []

print("Hyperparameter Tuning for C value:\n")
for C in C_values:
    model_tuned = LogisticRegression(C=C, random_state=42, solver='liblinear', max_iter=1000)
    model_tuned.fit(X_train, y_train)

    train_acc = model_tuned.score(X_train, y_train)
    train_accuracies.append(train_acc)

    val_acc = model_tuned.score(X_val, y_val)
    val_accuracies.append(val_acc)

    print(f"C: {C:<7.4f} | Training Accuracy: {train_acc:.4f} | Validation Accuracy: {val_acc:.4f}")

best_C_idx = np.argmax(val_accuracies)
best_C = C_values[best_C_idx]
best_val_accuracy = val_accuracies[best_C_idx]

print(f"\nBest C value found: {best_C:.4f} with Validation Accuracy: {best_val_accuracy:.4f}")

best_model = LogisticRegression(C=best_C, random_state=42, solver='liblinear', max_iter=1000)
best_model.fit(X_train, y_train)

print("Model retrained with the best C value on the training set (stored as `best_model`).")

Hyperparameter Tuning for C value:

C: 0.0001  | Training Accuracy: 0.8976 | Validation Accuracy: 0.8148
C: 0.0010  | Training Accuracy: 0.8976 | Validation Accuracy: 0.8148
C: 0.0100  | Training Accuracy: 0.8976 | Validation Accuracy: 0.8148
C: 0.1000  | Training Accuracy: 0.9449 | Validation Accuracy: 0.8889
C: 1.0000  | Training Accuracy: 0.9843 | Validation Accuracy: 0.8889
C: 10.0000 | Training Accuracy: 1.0000 | Validation Accuracy: 0.9259
C: 100.0000 | Training Accuracy: 1.0000 | Validation Accuracy: 0.9259

Best C value found: 10.0000 with Validation Accuracy: 0.9259
Model retrained with the best C value on the training set (stored as `best_model`).


## Testing

In [73]:
from sklearn.metrics import classification_report

test_accuracy = best_model.score(X_test, y_test)
print(f"Test Accuracy with best C ({best_C:.4f}): {test_accuracy:.4f}")

y_pred_test = best_model.predict(X_test)

report_test = classification_report(y_test, y_pred_test, target_names=best_model.classes_)
print("\nClassification Report (Test Data):\n", report_test)

model = best_model
print("\nGlobal 'model' variable updated to the best performing model (`best_model`).")

Test Accuracy with best C (10.0000): 0.9643

Classification Report (Test Data):
               precision    recall  f1-score   support

        DUCK       1.00      1.00      1.00         3
  ENTER_PIPE       1.00      1.00      1.00         2
        FIRE       0.75      1.00      0.86         3
        JUMP       1.00      0.67      0.80         3
   MOVE_LEFT       1.00      1.00      1.00         4
  MOVE_RIGHT       1.00      1.00      1.00         3
       PAUSE       1.00      1.00      1.00         3
         RUN       1.00      1.00      1.00         3
        STOP       1.00      1.00      1.00         4

    accuracy                           0.96        28
   macro avg       0.97      0.96      0.96        28
weighted avg       0.97      0.96      0.96        28


Global 'model' variable updated to the best performing model (`best_model`).


In [74]:
# longer test cases

test_cases = [
    "go backward left",
    "jump up high",
    "move super fast",
    "shoot him",
    "pause the game",
    "drop into pipe"
]

for t in test_cases:
    print(t, "→", predict(t))

go backward left → (np.str_('MOVE_LEFT'), np.float64(0.8524631346544648))
jump up high → (np.str_('JUMP'), np.float64(0.8347302720763576))
move super fast → (np.str_('RUN'), np.float64(0.6827374860279793))
shoot him → (np.str_('FIRE'), np.float64(0.7727963706838986))
pause the game → (np.str_('PAUSE'), np.float64(0.8881716786049116))
drop into pipe → (np.str_('ENTER_PIPE'), np.float64(0.7828419065672976))


In [75]:
import joblib

joblib.dump(model, "nlp_weights.joblib.pth")

['nlp_weights.joblib.pth']